<a href="https://colab.research.google.com/github/miraszadabek/Legal-Document-RAG/blob/main/202580018_Zadabek_Miras_NLLP_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U torch transformers accelerate bitsandbytes peft
!pip install -q -U langchain langchain-community langchain-huggingface chromadb
!pip install -q -U sentence-transformers datasets pypdf
!pip install -q -U streamlit pyngrok
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 793.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.

In [ ]:
!pip install -qU langchain-text-splitters

import os
import shutil
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Loading dataset...")
dataset = load_dataset("billsum", split="train[:200]")

documents = []
metadatas = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""]
)

print("Processing documents...")
for item in dataset:
    chunks = text_splitter.split_text(item['text'])
    for chunk in chunks:
        documents.append(chunk)
        metadatas.append({"title": item['title'], "summary": item['summary'][:100] + "..."})

print(f"Total chunks created: {len(documents)}")

# 3. Embeddings & Vector DB
print("Loading Embedding Model (MPNet)...")
embedding_model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}

hf_embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

persist_directory = "./chroma_db_advanced"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

print("Creating Vector Database...")
vectordb = Chroma.from_texts(
    texts=documents,
    embedding=hf_embeddings,
    metadatas=metadatas,
    persist_directory=persist_directory
)
print("Vector DB created and persisted successfully!")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating Vector Database...
Vector DB created and persisted successfully!


In [ ]:
%%writefile app.py
import streamlit as st
import torch
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

st.set_page_config(page_title="Legal AI Expert (Advanced RAG)", layout="wide")

st.title("⚖️ AI Legal Research Assistant")
st.markdown("""
**Architecture:** Hybrid Retrieval (Bi-Encoder + Cross-Encoder Reranking) powered by Quantized LLM.
This system analyzes legal documents from the BillSum dataset using a 2-stage retrieval process.
""")

@st.cache_resource
def load_resources():
    # 1. Embeddings
    emb_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )

    # 2. Vector DB
    vectordb = Chroma(persist_directory="./chroma_db_advanced", embedding_function=emb_model)

    # 3. Reranker (Cross-Encoder)
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')
    # 4. LLM (Generative Model) - Mistral/Zephyr in 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model_id = "HuggingFaceH4/zephyr-7b-beta"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

    text_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.95,
        top_k=40,
        repetition_penalty=1.1
    )

    return vectordb, reranker, text_pipeline

try:
    vectordb, reranker, llm_pipeline = load_resources()
    st.success("Neural Neural Networks loaded: Bi-Encoder, Cross-Encoder, and LLM (4-bit).")
except Exception as e:
    st.error(f"Error loading models: {e}")
    st.stop()

with st.sidebar:
    st.header("⚙️ RAG Parameters")
    retrieval_k = st.slider("Initial Retrieval (k)", 5, 20, 10)
    rerank_top_n = st.slider("Reranked Output (n)", 1, 5, 3)
    st.info("The system first retrieves 'k' documents, then the Cross-Encoder scores them and keeps the top 'n' for the LLM.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("Ask a question about the legal bills..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Генерация ответа
    with st.chat_message("assistant"):
        message_placeholder = st.empty()
        full_response = ""

        with st.spinner("🤖 Thinking (Retrieving -> Reranking -> Generating)..."):
            # 1. Retrieval (Stage 1)
            docs = vectordb.similarity_search(prompt, k=retrieval_k)
            doc_texts = [d.page_content for d in docs]
            pairs = [[prompt, doc_text] for doc_text in doc_texts]
            scores = reranker.predict(pairs)
            top_indices = scores.argsort()[::-1][:rerank_top_n]
            top_docs = [docs[i] for i in top_indices]
            context = ""
            sources = []
            for i, doc in enumerate(top_docs):
                context += f"[Document {i+1}]: {doc.page_content}\n\n"
                sources.append(f"**Doc {i+1}:** {doc.metadata['title']}")

            # 3. Generation (LLM)
            system_prompt = "You are a senior legal analyst. Use the provided context to answer the user's question accurately. If the answer is not in the context, state that you do not know. Cite the documents (e.g. [Document 1])."

            user_msg = f"Context:\n{context}\n\nQuestion: {prompt}"

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ]

            prompt_formatted = llm_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            outputs = llm_pipeline(prompt_formatted)
            generated_text = outputs[0]["generated_text"]
            response_clean = generated_text.split("<|assistant|>")[-1].strip()
            final_output = response_clean + "\n\n---\n**Sources utilized:**\n" + "\n".join(sources)

        message_placeholder.markdown(final_output)
        st.session_state.messages.append({"role": "assistant", "content": final_output})

Writing app.py


In [ ]:
%%writefile app.py
import streamlit as st
import torch
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

st.set_page_config(page_title="Legal AI Expert (Advanced RAG)", layout="wide")

st.title("⚖️ Enterprise AI Legal Assistant")
st.markdown("""
### System Architecture:
1.  **Retrieval:** Semantic Search via Bi-Encoder (MPNet).
2.  **Refinement:** Re-ranking top results via Cross-Encoder (MS-MARCO).
3.  **Generation:** 4-bit Quantized LLM (Zephyr-7B-Beta).
""")

@st.cache_resource
def load_resources():
    # 1. Embeddings
    emb_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )

    # 2. Vector DB
    vectordb = Chroma(persist_directory="./chroma_db_advanced", embedding_function=emb_model)

    # 3. Reranker (Cross-Encoder)
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

    # 4. LLM (Generative Model) - 4-bit Quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model_id = "HuggingFaceH4/zephyr-7b-beta"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

    text_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.4,
        top_p=0.95,
        top_k=40,
        repetition_penalty=1.15
    )

    return vectordb, reranker, text_pipeline

with st.spinner("🚀 Loading Neural Networks (Quantum LLM & Cross-Encoders)..."):
    try:
        vectordb, reranker, llm_pipeline = load_resources()
        st.success("✅ System Online: Models Loaded on GPU.")
    except Exception as e:
        st.error(f"Critical Error: {e}")
        st.stop()

with st.sidebar:
    st.header("⚙️ Hyperparameters")
    retrieval_k = st.slider("Stage 1: Retrieval Documents (k)", 5, 50, 15)
    rerank_top_n = st.slider("Stage 2: Reranked Documents (n)", 1, 10, 3)
    st.info("Stage 1 casts a wide net. Stage 2 uses a heavier model to filter only the best context.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("Enter your legal query here..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        message_placeholder = st.empty()

        with st.spinner("🤖 Analyzing documents..."):
            # 1. Retrieval (Stage 1)
            docs = vectordb.similarity_search(prompt, k=retrieval_k)

            # 2. Reranking (Stage 2) - Advanced Step
            doc_texts = [d.page_content for d in docs]
            pairs = [[prompt, doc_text] for doc_text in doc_texts]
            scores = reranker.predict(pairs)
            top_indices = scores.argsort()[::-1][:rerank_top_n]
            top_docs = [docs[i] for i in top_indices]
            context_str = ""
            sources_list = []
            for i, doc in enumerate(top_docs):
                clean_content = doc.page_content.replace("\n", " ")
                context_str += f"[Source ID: {i+1}]\nText: {clean_content}\n\n"
                sources_list.append(f"**Source {i+1}:** {doc.metadata['title']}")

            # 3. Generation (LLM)
            system_instruction = (
                "You are an expert legal assistant. Answer the user's question primarily based on the provided Context. "
                "If the answer is not in the context, say 'I cannot find the answer in the provided documents'. "
                "Cite sources using [Source ID: X] format."
            )

            user_msg = f"Context:\n{context_str}\n\nQuestion: {prompt}"

            messages = [
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_msg},
            ]

            prompt_formatted = llm_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            outputs = llm_pipeline(prompt_formatted)
            raw_output = outputs[0]["generated_text"]
            final_answer = raw_output.split("<|assistant|>")[-1].strip()
            display_text = final_answer + "\n\n---\n**📚 Verified Sources:**\n" + "\n".join(sources_list)

        message_placeholder.markdown(display_text)
        st.session_state.messages.append({"role": "assistant", "content": display_text})

Overwriting app.py


In [ ]:
import os
from pyngrok import ngrok

my_token = "36e8DHR3WcTKiBTBVAZeYUEbnKf_7B6ttF4XqbXW6E5rPoVJP"

ngrok.set_auth_token(my_token)

ngrok.kill()

public_url = ngrok.connect(8501).public_url
print(f" Твое стабильное приложение здесь: {public_url}")

!streamlit run app.py

 Твое стабильное приложение здесь: https://unpoached-watertight-lacresha.ngrok-free.dev



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.199.175.218:8501

2025-12-11 11:13:16.951652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765451597.003151    2788 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765451597.017033    2788 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765451597.051783    2788 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than onc